# P4AI — Deep learning on Animals-10

**Runtime:** Colab, T4 GPU.
The full run takes several hours (E1/E4 train 20 epochs from scratch; E8/E9 answer ~4k test images
one by one). To run a subset, edit `RUN_EXPS` in section 1.

| | model | init | protocol |
|---|---|---|---|
| E1 | ConvNeXt-T | from scratch | 20 epochs |
| E2 | ConvNeXt-T | ImageNet-22k (timm) | fine-tune 5 epochs |
| E3 | ConvNeXt-T | DINOv3 (LVD-1689M) | fine-tune 5 epochs |
| E4 | ViT-S/16 | from scratch | 20 epochs |
| E5 | ViT-S/16 | ImageNet-21k (timm) | fine-tune 5 epochs |
| E6 | ViT-S/16 | DINOv3 (LVD-1689M) | fine-tune 5 epochs |
| E7 | MobileCLIP2-S0 | zero-shot | image-text similarity |
| E8 | Qwen3.5-0.8B | zero-shot | generative VLM: prompt + one-word answer |
| E9 | Valen-Decision-1.5-0.8B | zero-shot | decision model: choice over the 10 classes |

Protocol
* Same stratified 70 / 15 / 15 split (seed 42), no data augmentation.
* Test metrics are reported for the FINAL epoch (no checkpoint selection); the best-validation epoch
  is only logged.
* Zero-shot models: a 500-image stratified validation subset is used only to choose the prompt /
  templates and to fit a temperature; the test split is scored once.
* E8/E9 receive the same 224x224 resize as the CNN/ViT loader; E7 uses its own preprocessing.

# 0. Setup

In [ ]:
import os, sys, json, time, math, random, glob, gc, re, subprocess, warnings, hashlib
from collections import defaultdict


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


pip_install("-U", "timm", "open_clip_torch", "transformers", "accelerate", "kagglehub", "av")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("timm", timm.__version__, "| torch", torch.__version__)

# 1. Configuration

In [ ]:
SEED = 42
NUM_CLASSES = 10
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 5
WARMUP_EPOCHS = 1
LABEL_SMOOTHING = 0.1            # loss regularizer, not data augmentation; set 0.0 to disable
WEIGHT_DECAY = 0.05
NUM_WORKERS = 2

RUN_EXPS = ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9"]    # experiments to run

WORK_DIR = "/content/p4ai_dl"     # local Colab disk: trained weights + resized images for E8/E9
ZS_IMG_SIZE = 224                 # E8/E9 input size (None = original images)
ZS_VAL_SUBSET = 500               # stratified validation images for prompt choice + temperature

VIZ_N_TEST = 6
VIZ_SEED = SEED + 1337

CLIP_NAME, CLIP_TAG = "MobileCLIP2-S0", "dfndr2b"
QWEN_ID = "Qwen/Qwen3.5-0.8B"
VALEN_ID = "Valen-Team/Valen-Decision-1.5-0.8B"

TRANSLATE = {"cane": "dog", "cavallo": "horse", "elefante": "elephant", "farfalla": "butterfly",
             "gallina": "chicken", "gatto": "cat", "mucca": "cow", "pecora": "sheep",
             "ragno": "spider", "scoiattolo": "squirrel"}

TRAINED = {   # Group A: lr = (backbone, head); "epochs" overrides NUM_EPOCHS for that run
    "E1": dict(name="ConvNeXt-T scratch", family="CNN", init="scratch",
               timm="convnext_tiny", pretrained=False, lr=(1e-3, 1e-3), epochs=20),
    "E2": dict(name="ConvNeXt-T IN-22k", family="CNN", init="supervised",
               timm="convnext_tiny.fb_in22k_ft_in1k", pretrained=True, lr=(1e-4, 1e-3)),
    "E3": dict(name="ConvNeXt-T DINOv3", family="CNN", init="DINOv3",
               timm="convnext_tiny.dinov3_lvd1689m", pretrained=True, lr=(1e-4, 1e-3)),
    "E4": dict(name="ViT-S/16 scratch", family="ViT", init="scratch",
               timm="vit_small_patch16_224", pretrained=False, lr=(1e-3, 1e-3), epochs=20),
    "E5": dict(name="ViT-S/16 IN-21k", family="ViT", init="supervised",
               timm="vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True, lr=(5e-5, 1e-3)),
    "E6": dict(name="ViT-S/16 DINOv3", family="ViT", init="DINOv3",
               timm="vit_small_patch16_dinov3.lvd1689m", pretrained=True, lr=(5e-5, 1e-3)),
}
ZERO_SHOT = {
    "E7": dict(name="MobileCLIP2-S0 zero-shot", family="Embedding", init="zero-shot"),
    "E8": dict(name="Qwen3.5-0.8B zero-shot", family="Generative VLM", init="zero-shot"),
    "E9": dict(name="Valen-Decision-1.5-0.8B zero-shot", family="Decision model", init="zero-shot"),
}
EXPS = {**TRAINED, **ZERO_SHOT}

MODEL_DIR = os.path.join(WORK_DIR, "models")
ZS_CACHE = os.path.join(WORK_DIR, "zs_224")
for _d in (WORK_DIR, MODEL_DIR, ZS_CACHE):
    os.makedirs(_d, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


seed_everything()

# 2. Data and split

In [ ]:
def get_animals10():
    """Download Animals-10 with kagglehub (cached) and return the path to raw-img/."""
    try:
        import kagglehub
    except ImportError:
        pip_install("kagglehub"); import kagglehub
    root = kagglehub.dataset_download("alessiocorrado99/animals10")
    hits = sorted(glob.glob(os.path.join(root, "**", "raw-img"), recursive=True))
    if not hits:
        raise RuntimeError(f"raw-img/ not found under {root}")
    return hits[0].rstrip("/") + "/"


DATA_DIR = get_animals10()
print("DATA_DIR:", DATA_DIR)


class AnimalDataset(Dataset):
    """ImageFolder subset with translated class names. transform=None returns the PIL image."""
    def __init__(self, root, indices=None, transform=None, translate_map=None):
        self.base_ds = ImageFolder(root)
        self.transform = transform
        self.translate_map = translate_map or {}
        self.idx_to_class = {v: self.translate_map.get(k, k) for k, v in self.base_ds.class_to_idx.items()}
        self.indices = list(indices) if indices is not None else list(range(len(self.base_ds)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img_path, label = self.base_ds.samples[self.indices[i]]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    @property
    def classes(self):
        return [self.idx_to_class[i] for i in range(NUM_CLASSES)]


def make_splits(root, train_ratio=0.70, val_ratio=0.15, seed=SEED):
    """Stratified 70/15/15 split (identical to the classical-ML notebook)."""
    base = ImageFolder(root)
    labels = [s[1] for s in base.samples]
    indices = list(range(len(labels)))
    train_idx, temp_idx, _, y_temp = train_test_split(
        indices, labels, test_size=1 - train_ratio, stratify=labels, random_state=seed)
    rel_val = val_ratio / (1 - train_ratio)
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=1 - rel_val, stratify=y_temp, random_state=seed)
    print(f"Split sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
    return train_idx, val_idx, test_idx


def get_transforms(timm_name):
    """Resize to 224x224 + the model's normalization (no augmentation)."""
    model = timm.create_model(timm_name, pretrained=False)
    cfg = timm.data.resolve_data_config({}, model=model)
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=cfg["mean"], std=cfg["std"])])


def build_loaders(root, train_idx, val_idx, test_idx, timm_name):
    tf = get_transforms(timm_name)
    train_ds = AnimalDataset(root, train_idx, transform=tf, translate_map=TRANSLATE)
    val_ds = AnimalDataset(root, val_idx, transform=tf, translate_map=TRANSLATE)
    test_ds = AnimalDataset(root, test_idx, transform=tf, translate_map=TRANSLATE)
    g = torch.Generator().manual_seed(SEED)
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g,
                              num_workers=NUM_WORKERS, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    return train_loader, val_loader, test_loader


train_idx, val_idx, test_idx = make_splits(DATA_DIR)
TRAIN_IDX, VAL_IDX, TEST_IDX = map(np.asarray, (train_idx, val_idx, test_idx))
BASE = ImageFolder(DATA_DIR)
SAMPLES = BASE.samples
LABELS = np.array([l for _, l in SAMPLES])
CLASS_NAMES = AnimalDataset(DATA_DIR, translate_map=TRANSLATE).classes
Y_VAL, Y_TEST = LABELS[VAL_IDX], LABELS[TEST_IDX]
print("Classes:", CLASS_NAMES)

# Stratified validation subset used by the zero-shot runs.
_rng = np.random.default_rng(SEED)
ZS_VAL_IDX = np.sort(np.concatenate([
    _rng.choice(VAL_IDX[Y_VAL == c], max(1, ZS_VAL_SUBSET // NUM_CLASSES), replace=False)
    for c in range(NUM_CLASSES)]))
Y_ZS_VAL = LABELS[ZS_VAL_IDX]

# 3. Shared utilities: metrics, calibration, results

In [ ]:
def cls_metrics(y, pred):
    """Macro metrics over the 10 classes. pred = -1 (invalid answer) counts as wrong."""
    labels = list(range(NUM_CLASSES))
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=labels, average="macro", zero_division=0)
    f_cls = precision_recall_fscore_support(y, pred, labels=labels, average=None, zero_division=0)[2]
    return {"acc": accuracy_score(y, pred), "prec": p, "rec": r, "f1": f}, f_cls


def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


def fit_temperature(logits, y):
    """Temperature scaling (Guo et al., 2017): one scalar T fitted by NLL on validation logits."""
    lg = torch.tensor(np.asarray(logits), dtype=torch.float64)
    yt = torch.tensor(np.asarray(y))
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(lg / log_t.exp(), yt)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp())


def softmax_np(z, t=1.0):
    z = np.asarray(z, dtype=np.float64) / t
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def log_probs(p):
    return np.log(np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0))


def model_path(eid):
    return os.path.join(MODEL_DIR, f"{eid}.pth")


def epochs_of(eid):
    return TRAINED[eid].get("epochs", NUM_EPOCHS)


RESULTS = {}        # eid -> result row
PREDS = {}          # eid -> test predictions, probabilities, per-class F1, ...
HISTORY = {}        # eid -> training curves (E1-E6)


def record_run(eid, test_logits, row, y_pred=None, val_logits=None, val_y=None, **arrays):
    """test_logits: (n_test, 10) logits or log-probabilities. The temperature is fitted on
    (val_logits, val_y) when given. Stores the result row and the test predictions."""
    probs = softmax_np(test_logits)
    y_pred = probs.argmax(1) if y_pred is None else np.asarray(y_pred)
    m, f_cls = cls_metrics(Y_TEST, y_pred)
    temp = fit_temperature(val_logits, val_y) if val_logits is not None else 1.0
    probs_ts = softmax_np(test_logits, temp)
    RESULTS[eid] = {"eid": eid, "name": EXPS[eid]["name"], "family": EXPS[eid]["family"],
                    "init": EXPS[eid]["init"], **{f"test_{k}": v for k, v in m.items()},
                    "ece": ece_score(probs, Y_TEST), "ece_ts": ece_score(probs_ts, Y_TEST),
                    "temperature": temp, **row}
    PREDS[eid] = {"y_true": Y_TEST, "y_pred": y_pred, "probs": probs, "probs_ts": probs_ts,
                  "f1_per_class": f_cls, **arrays}
    print(f"[{eid}] test acc {m['acc']:.4f} | macro-F1 {m['f1']:.4f} | "
          f"ECE {RESULTS[eid]['ece']:.4f} -> {RESULTS[eid]['ece_ts']:.4f} (T={temp:.2f})")


def run_row(eid):
    return RESULTS[eid]


def map_progress(tag, items, fn):
    """[fn(x) for x in items] with a progress bar. Returns (results, seconds per item)."""
    t0 = time.time()
    out = [fn(x) for x in tqdm(items, desc=tag)]
    return out, (time.time() - t0) / max(len(items), 1)


def n_params(module):
    return sum(p.numel() for p in module.parameters()) / 1e6


class managed_loader:
    """Iterate a multi-worker DataLoader safely in a notebook: collect stale iterators first,
    and always shut the workers down on exit (also when interrupted)."""
    def __init__(self, dl, desc):
        self.dl, self.desc = dl, desc

    def __enter__(self):
        gc.collect()
        self.it = iter(self.dl)
        return tqdm(self.it, total=len(self.dl), desc=self.desc, leave=False)

    def __exit__(self, *exc):
        shutdown = getattr(self.it, "_shutdown_workers", None)
        if shutdown is not None:
            shutdown()
        self.it = None
        return False


def free_gpu():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()


def peak_vram():
    return torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else float("nan")


def autocast():
    return torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

# 4. Group A: ConvNeXt-T / ViT-S, no augmentation
E1/E4 from scratch (20 epochs); E2/E3/E5/E6 fine-tuned from pretrained weights (5 epochs).
AdamW, warm-up + cosine schedule, mixed precision; the test set is scored with the final weights.

In [ ]:
class AnimalClassifier(nn.Module):
    """timm backbone (no classifier) + MLP head."""
    def __init__(self, timm_name, n_classes, pretrained=True, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        in_features = self.get_in_features()
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(hidden_dim, n_classes))

    def get_in_features(self):
        self.backbone.eval()
        with torch.no_grad():
            return self.backbone(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE)).shape[-1]

    def forward(self, x):
        return self.head(self.backbone(x))


def warmup_cosine(optimizer, steps_per_epoch, epochs, warmup_epochs=WARMUP_EPOCHS):
    total, warm = steps_per_epoch * epochs, steps_per_epoch * warmup_epochs

    def factor(step):
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, desc):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    with managed_loader(loader, desc) as batches:
        for x, y in batches:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, desc):
    model.eval()
    logits, ys, loss_sum = [], [], 0.0
    with managed_loader(loader, desc) as batches:
        for x, y in batches:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                out = model(x)
            loss_sum += criterion(out.float(), y).item() * len(y)
            logits.append(out.float().cpu()); ys.append(y.cpu())
    logits, ys = torch.cat(logits).numpy(), torch.cat(ys).numpy()
    return loss_sum / len(ys), logits, ys


def run_trained(eid):
    cfg = TRAINED[eid]
    n_ep = epochs_of(eid)
    print(f"\n{'=' * 70}\n {eid}: {cfg['name']} ({cfg['timm']}, pretrained={cfg['pretrained']}, "
          f"{n_ep} epochs)\n{'=' * 70}")
    seed_everything(); free_gpu()
    train_loader, val_loader, test_loader = build_loaders(DATA_DIR, train_idx, val_idx, test_idx, cfg["timm"])
    model = AnimalClassifier(cfg["timm"], NUM_CLASSES, pretrained=cfg["pretrained"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    lr_b, lr_h = cfg["lr"]
    optimizer = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": lr_b},
                                   {"params": model.head.parameters(), "lr": lr_h}],
                                  weight_decay=WEIGHT_DECAY)
    scheduler = warmup_cosine(optimizer, len(train_loader), n_ep)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    hist, train_s = defaultdict(list), 0.0
    for ep in range(1, n_ep + 1):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler,
                                 f"{eid} train ep{ep}/{n_ep}")
        vl, vlog, vy = evaluate(model, val_loader, criterion, f"{eid} val ep{ep}")
        vm, _ = cls_metrics(vy, vlog.argmax(1))
        for k, v in {"epoch": ep, "train_loss": tl, "train_acc": ta, "val_loss": vl,
                     "val_acc": vm["acc"], "val_f1": vm["f1"], "epoch_s": time.time() - t0,
                     "lr_backbone": optimizer.param_groups[0]["lr"]}.items():
            hist[k].append(v)
        train_s += time.time() - t0
        print(f"  ep {ep}: train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {vm['acc']:.4f} "
              f"F1 {vm['f1']:.4f} | gap {ta - vm['acc']:+.4f} | {time.time() - t0:.0f}s")

    # Test with the FINAL weights (no early stopping, no checkpoint selection).
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    _, test_logits, ty = evaluate(model, test_loader, criterion, f"{eid} test")
    test_s = time.time() - t0
    assert np.array_equal(ty, Y_TEST)
    torch.save({"eid": eid, "timm": cfg["timm"], "state_dict": model.state_dict()}, model_path(eid))
    HISTORY[eid] = dict(hist)
    best = int(np.argmax(hist["val_f1"]))
    record_run(eid, test_logits,
               {"model_id": cfg["timm"], "params_M": n_params(model), "epochs": n_ep,
                "augmentation": False, "final_val_acc": hist["val_acc"][-1], "final_val_f1": hist["val_f1"][-1],
                "best_val_epoch": best + 1, "best_val_f1": hist["val_f1"][best],
                "final_train_acc": hist["train_acc"][-1], "train_s": train_s,
                "test_img_per_s": len(Y_TEST) / test_s, "peak_vram_gb": peak_vram()},
               val_logits=vlog, val_y=vy)
    del model, optimizer, train_loader, val_loader, test_loader
    free_gpu()


for eid in TRAINED:
    if eid in RUN_EXPS:
        run_trained(eid)

# 5. Group B: zero-shot models (E7-E9)
Each model is wrapped in a small predictor class shared by the runs below and section 7.

In [ ]:
def zs_image_paths(indices):
    """Paths for E8/E9: the 224x224 resize of the CNN/ViT loader, cached as PNG (lossless)."""
    if ZS_IMG_SIZE is None:
        return [os.path.abspath(SAMPLES[i][0]) for i in indices]
    resize = transforms.Resize((ZS_IMG_SIZE, ZS_IMG_SIZE))
    out = []
    for i in indices:
        dst = os.path.join(ZS_CACHE, f"{int(i)}.png")
        if not os.path.exists(dst):
            resize(Image.open(SAMPLES[i][0]).convert("RGB")).save(dst)
        out.append(os.path.abspath(dst))
    return out

## 5.1 E7 — MobileCLIP2-S0 (embedding, image-text similarity)
Prompt templates and class-name wording are chosen on the validation subset.

In [ ]:
CLIP_TEMPLATE_SETS = {
    "single": ["a photo of a {}."],
    "ensemble3": ["a photo of a {}.", "a close-up photo of a {}.", "a photo of a {}, a type of animal."],
    "ensemble6": ["a photo of a {}.", "a close-up photo of a {}.", "a photo of a {}, a type of animal.",
                  "a blurry photo of a {}.", "a photo of the {} in the wild.", "a cropped photo of a {}."],
}
CLIP_NAME_SETS = {
    "plain": None,                                   # the class names as they are
    "clarified": {"chicken": "chicken, a bird", "cow": "cow, a farm animal",
                  "spider": "spider, an arachnid", "butterfly": "butterfly, an insect"},
}
E7_CHOICE = None      # {"templates": ..., "names": ...}, set by run_e7()


def e7_text_config():
    return E7_CHOICE


class ClipPredictor:
    def __init__(self):
        import open_clip
        from timm.utils.model import reparameterize_model
        m, _, self.preprocess = open_clip.create_model_and_transforms(CLIP_NAME, pretrained=CLIP_TAG)
        self.model = reparameterize_model(m.eval()).to(DEVICE)
        self.tokenizer = open_clip.get_tokenizer(CLIP_NAME)
        self.scale = float(self.model.logit_scale.exp())
        self.W = None
        c = e7_text_config()
        if c is not None:
            self.set_text(c["templates"], c["names"])

    @torch.no_grad()
    def set_text(self, template_key, names_key):
        names = [(CLIP_NAME_SETS[names_key] or {}).get(c, c) for c in CLASS_NAMES]
        W = []
        for n in names:
            toks = self.tokenizer([t.format(n) for t in CLIP_TEMPLATE_SETS[template_key]]).to(DEVICE)
            e = F.normalize(self.model.encode_text(toks).float(), dim=-1).mean(0)
            W.append(F.normalize(e, dim=0))
        self.W = torch.stack(W)

    @torch.no_grad()
    def image_features(self, indices):
        ds = AnimalDataset(DATA_DIR, list(indices), transform=self.preprocess, translate_map=TRANSLATE)
        dl = DataLoader(ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS)
        feats = []
        with managed_loader(dl, "MobileCLIP2 images") as batches:
            for x, _ in batches:
                with autocast():
                    f = self.model.encode_image(x.to(DEVICE))
                feats.append(F.normalize(f.float(), dim=-1))
        return torch.cat(feats)

    def logits(self, feats):
        return (self.scale * feats @ self.W.T).cpu().numpy()


def run_e7():
    global E7_CHOICE
    free_gpu()
    clip = ClipPredictor()
    val_f = clip.image_features(ZS_VAL_IDX)
    search = []
    for tk in CLIP_TEMPLATE_SETS:
        for nk in CLIP_NAME_SETS:
            clip.set_text(tk, nk)
            m, _ = cls_metrics(Y_ZS_VAL, clip.logits(val_f).argmax(1))
            search.append({"templates": tk, "names": nk, "val_f1": m["f1"], "val_acc": m["acc"]})
    search = pd.DataFrame(search).sort_values("val_f1", ascending=False)
    print("Template / class-name choice on the validation subset:\n", search.to_string(index=False))
    best = search.iloc[0]
    E7_CHOICE = {"templates": best["templates"], "names": best["names"]}
    clip.set_text(best["templates"], best["names"])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    test_f = clip.image_features(TEST_IDX)
    test_s = time.time() - t0
    record_run("E7", clip.logits(test_f),
               {"model_id": f"{CLIP_NAME} ({CLIP_TAG})", "params_M": n_params(clip.model),
                "prompt": f"{best['templates']} / {best['names']}", "zs_val_f1": best["val_f1"],
                "test_img_per_s": len(TEST_IDX) / test_s, "peak_vram_gb": peak_vram()},
               val_logits=clip.logits(val_f), val_y=Y_ZS_VAL)
    del clip; free_gpu()


if "E7" in RUN_EXPS:
    run_e7()

## 5.2 E8 — Qwen3.5-0.8B (generative VLM: prompt + generated answer)
* The headline prediction is the generated word, mapped to a class (unmappable answers count as
  wrong and are reported as the invalid rate).
* For calibration and the visualizations, the 10-class probabilities come from the logits of the
  FIRST generated token, restricted to the first token of each class name.

In [ ]:
QWEN_PROMPTS = {
    "list": "What animal is in this image? Answer with exactly one word from this list: {classes}.",
    "classify": "Classify the animal in this photo. Possible answers: {classes}. Reply with the answer only.",
}
SYNONYMS = {"hen": "chicken", "rooster": "chicken", "chick": "chicken", "kitten": "cat", "puppy": "dog",
            "lamb": "sheep", "calf": "cow", "cattle": "cow", "bull": "cow", "pony": "horse", "foal": "horse"}


def parse_answer(text):
    """Map a generated answer to a class index, or -1 if it names none of the classes."""
    for w in re.findall(r"[a-z]+", text.lower()):
        for cand in (w, w[:-1] if w.endswith("s") else w, SYNONYMS.get(w)):
            if cand in CLASS_NAMES:
                return CLASS_NAMES.index(cand)
    return -1


def show_token(s):
    """Make whitespace in a decoded token visible."""
    s = s.replace(" ", "▁").replace("\n", "↵")
    return s if s else "∅"


class QwenPredictor:
    def __init__(self):
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.proc = AutoProcessor.from_pretrained(QWEN_ID)
        self.model = AutoModelForImageTextToText.from_pretrained(QWEN_ID, dtype=torch.float32).to(DEVICE).eval()
        self.class_ids = self._class_token_ids()

    def _class_token_ids(self):
        tok = self.proc.tokenizer
        cand = {i: {tok.encode(v, add_special_tokens=False)[0]
                    for v in (c, c.capitalize(), " " + c, " " + c.capitalize())}
                for i, c in enumerate(CLASS_NAMES)}
        owners = defaultdict(set)
        for i, s in cand.items():
            for t in s:
                owners[t].add(i)
        ids = {}
        for i, s in cand.items():
            own = sorted(t for t in s if len(owners[t]) == 1)
            if not own:
                print(f"  WARNING: '{CLASS_NAMES[i]}' shares all first tokens with another class")
                own = sorted(s)
            ids[i] = own
        return ids

    def inputs(self, path, prompt_key):
        prompt = QWEN_PROMPTS[prompt_key].format(classes=", ".join(CLASS_NAMES))
        msgs = [{"role": "user", "content": [{"type": "image", "path": path}, {"type": "text", "text": prompt}]}]
        return self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True,
                                             return_tensors="pt", enable_thinking=False).to(DEVICE)

    @torch.no_grad()
    def ask(self, path, prompt_key="list"):
        """Generated answer + log-probabilities of the 10 classes at the first answer token."""
        inp = self.inputs(path, prompt_key)
        out = self.model.generate(**inp, max_new_tokens=8, do_sample=False,
                                  output_logits=True, return_dict_in_generate=True)
        text = self.proc.decode(out.sequences[0, inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        lp = torch.log_softmax(out.logits[0][0].float(), -1)
        class_lp = torch.stack([torch.logsumexp(lp[self.class_ids[i]], 0) for i in range(NUM_CLASSES)])
        return text, torch.log_softmax(class_lp, 0).cpu().numpy()

    @torch.no_grad()
    def ask_detailed(self, path, prompt_key="list", topk=12):
        """Same generation as ask(), plus the full-vocabulary distribution of the first answer token
        (top-k), the share of it on class-name tokens, and the probability of every generated token."""
        inp = self.inputs(path, prompt_key)
        out = self.model.generate(**inp, max_new_tokens=8, do_sample=False,
                                  output_logits=True, return_dict_in_generate=True)
        tok = self.proc.tokenizer
        gen_ids = out.sequences[0, inp["input_ids"].shape[1]:].tolist()
        steps = []
        for t, tid in enumerate(gen_ids):
            p = torch.softmax(out.logits[t][0].float(), -1)
            top = torch.topk(p, 3)
            steps.append({"token": show_token(tok.decode([tid])), "prob": float(p[tid]),
                          "alts": [(show_token(tok.decode([int(i)])), float(v))
                                   for v, i in zip(top.values, top.indices)]})
        p0 = torch.softmax(out.logits[0][0].float(), -1)
        id2class = {t: i for i, ids in self.class_ids.items() for t in ids}
        top = torch.topk(p0, topk)
        first = [{"token": show_token(tok.decode([int(i)])), "prob": float(v), "cls": id2class.get(int(i), -1)}
                 for v, i in zip(top.values, top.indices)]
        class_mass = [float(p0[self.class_ids[i]].sum()) for i in range(NUM_CLASSES)]
        text = self.proc.decode(gen_ids, skip_special_tokens=True).strip()
        return {"text": text, "pred": parse_answer(text), "steps": steps, "first_topk": first,
                "class_probs": np.array(class_mass) / max(sum(class_mass), 1e-12),
                "class_mass": float(sum(class_mass)), "n_input_tokens": int(inp["input_ids"].shape[1])}

    @torch.no_grad()
    def class_logprobs(self, path, prompt_key):
        """One forward pass (no generation): log-probability of each class's first answer token."""
        out = self.model(**self.inputs(path, prompt_key), logits_to_keep=1)
        lp = torch.log_softmax(out.logits[0, -1].float(), -1)
        return torch.stack([torch.logsumexp(lp[self.class_ids[i]], 0) for i in range(NUM_CLASSES)]).cpu().numpy()


def run_e8():
    free_gpu()
    qwen = QwenPredictor()
    val_paths = zs_image_paths(ZS_VAL_IDX)
    search, val_out = [], {}
    for pk in QWEN_PROMPTS:
        res_, _ = map_progress(f"E8 val ({pk})", val_paths, lambda p: qwen.ask(p, pk))
        pred = np.array([parse_answer(t) for t, _ in res_])
        m, _ = cls_metrics(Y_ZS_VAL, pred)
        search.append({"prompt": pk, "val_f1": m["f1"], "val_acc": m["acc"], "invalid": int((pred < 0).sum())})
        val_out[pk] = np.stack([lp for _, lp in res_])
    search = pd.DataFrame(search).sort_values("val_f1", ascending=False)
    print("Prompt choice on the validation subset:\n", search.to_string(index=False))
    best = search.iloc[0]["prompt"]
    res_, s_per = map_progress(f"E8 test ({best})", zs_image_paths(TEST_IDX), lambda p: qwen.ask(p, best))
    texts = np.array([t for t, _ in res_])
    pred = np.array([parse_answer(t) for t in texts])
    test_lp = np.stack([lp for _, lp in res_])
    record_run("E8", test_lp,
               {"model_id": QWEN_ID, "params_M": n_params(qwen.model), "prompt": best,
                "zs_val_f1": float(search.iloc[0]["val_f1"]), "invalid_rate": float((pred < 0).mean()),
                "scored_vs_generated_agree": float((test_lp.argmax(1) == pred).mean()),
                "scored_test_acc": float((test_lp.argmax(1) == Y_TEST).mean()),
                "test_img_per_s": 1 / s_per, "peak_vram_gb": peak_vram()},
               y_pred=pred, val_logits=val_out[best], val_y=Y_ZS_VAL, answers=texts)
    print("Most common invalid answers:", pd.Series(texts[pred < 0]).value_counts().head(10).to_dict())
    del qwen; free_gpu()


if "E8" in RUN_EXPS:
    run_e8()

## 5.3 E9 — Valen-Decision-1.5-0.8B (System 1 decision model)
One `choice` question per image; the model returns a probability for every class in one forward
pass. Each image is asked twice, with the classes in two different orders, and the two probability
vectors are averaged. The model's remote code runs with `trust_remote_code=True`.

In [ ]:
VALEN_QUESTION = "Which animal is shown in the image?"
VALEN_ORDERS = {"original": list(CLASS_NAMES),
                "shuffled": [CLASS_NAMES[i] for i in np.random.default_rng(SEED).permutation(NUM_CLASSES)]}


def skip_flash_attn_import_check():
    """Valen imports flash_attn only if it is installed and otherwise uses PyTorch SDPA, but
    transformers' pre-load import scan flags the optional import. Do NOT install flash_attn
    on a T4: FlashAttention-2 needs Ampere or newer."""
    import transformers.dynamic_module_utils as dmu
    if getattr(dmu.get_imports, "_skips_flash", False):
        return
    orig = dmu.get_imports

    def get_imports(filename):
        return [i for i in orig(filename) if i != "flash_attn"]
    get_imports._skips_flash = True
    dmu.get_imports = get_imports


class ValenPredictor:
    def __init__(self):
        from transformers import AutoModel
        skip_flash_attn_import_check()
        self.model = AutoModel.from_pretrained(VALEN_ID, trust_remote_code=True, dtype=torch.float32,
                                               attn_implementation="sdpa").to(DEVICE).eval()
        # The T4 only emulates bf16: use FP32 compute instead of the checkpoint's bf16 autocast.
        self.model.backbone_autocast_dtype = None

    def ask_raw(self, path, order=None):
        order = order or CLASS_NAMES
        req = {"state": {"messages": [{"role": "user", "content": [
                   {"type": "image_url", "image_url": {"url": path}}]}]},
               "questions": {"animal": {"type": "choice", "instructions": VALEN_QUESTION,
                                        "criteria": {c: f"A {c}" for c in order}}}}
        return self.model.predict(req, temperature=1.0)

    def ask(self, path, order=None):
        ans = self.ask_raw(path, order)["answers"]["animal"]
        return np.array([ans["probabilities"][c] for c in CLASS_NAMES], dtype=np.float64)


def run_e9():
    free_gpu()
    valen = ValenPredictor()
    per_order, s_list = {}, []
    for split, idx in (("val", ZS_VAL_IDX), ("test", TEST_IDX)):
        paths = zs_image_paths(idx)
        for ok, order in VALEN_ORDERS.items():
            res_, s_per = map_progress(f"E9 {split} ({ok} order)", paths, lambda p: valen.ask(p, order))
            per_order[(split, ok)] = np.stack(res_)
            if split == "test":
                s_list.append(s_per)
    avg = {s: np.mean([per_order[(s, o)] for o in VALEN_ORDERS], 0) for s in ("val", "test")}
    a, b = (per_order[("test", o)].argmax(1) for o in VALEN_ORDERS)
    acc_by_order = {o: float((per_order[("test", o)].argmax(1) == Y_TEST).mean()) for o in VALEN_ORDERS}
    print("Test accuracy per option order:", acc_by_order)
    record_run("E9", log_probs(avg["test"]),
               {"model_id": VALEN_ID, "params_M": n_params(valen.model), "prompt": "choice x2 orders (avg)",
                "order_flip_rate": float((a != b).mean()), "acc_order_original": acc_by_order["original"],
                "acc_order_shuffled": acc_by_order["shuffled"],
                "test_img_per_s": 1 / np.mean(s_list), "peak_vram_gb": peak_vram()},
               val_logits=log_probs(avg["val"]), val_y=Y_ZS_VAL,
               **{f"probs_{o}": per_order[("test", o)] for o in VALEN_ORDERS})
    del valen; free_gpu()


if "E9" in RUN_EXPS:
    run_e9()

# 6. Results

In [ ]:
plt.rcParams.update({"savefig.dpi": 200, "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})


def show_figure(fig):
    if fig is not None:
        display(fig); plt.close(fig)


def show_table(df):
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 250,
                           "display.max_columns", 60):
        display(df)
    return df


def macro_f1_fast(y, p, k=10):
    ok = p >= 0
    cm = np.bincount(y[ok] * k + p[ok], minlength=k * k).reshape(k, k)
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, np.bincount(y, minlength=k) - tp
    return float((2 * tp / np.maximum(2 * tp + fp + fn, 1)).mean())


def bootstrap_ci(y, p, n=1000, seed=42):
    """95% percentile bootstrap CI of the test macro-F1 (1000 resamples of the test set)."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), (n, len(y)))
    s = np.array([macro_f1_fast(y[i], p[i]) for i in idx])
    return np.percentile(s, [2.5, 97.5])


def confusion_panel(ax, y, p, class_names, title):
    """Row-normalized confusion matrix; predictions < 0 (invalid answers) are left out of the columns."""
    k = len(class_names)
    ok = p >= 0
    cm = np.bincount(y[ok] * k + p[ok], minlength=k * k).reshape(k, k)
    cm = cm / np.maximum(np.bincount(y, minlength=k)[:, None], 1)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(k):
        for j in range(k):
            if cm[i, j] >= 0.01:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cm[i, j] > 0.5 else "black")
    ax.set_xticks(range(k)); ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(k)); ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel("predicted", fontsize=8); ax.set_ylabel("true", fontsize=8)
    ax.set_title(title, fontsize=9)


def fig_confusion_one(y, p, class_names, title):
    fig, ax = plt.subplots(figsize=(3.7, 3.5))
    confusion_panel(ax, np.asarray(y), np.asarray(p), class_names, title)
    fig.tight_layout()
    return fig


def heatmap_panel(ax, values, row_labels, col_labels, cmap="RdYlGn", vmin=None, vmax=100, fmt="{:.0f}"):
    vmin = max(0, np.nanmin(values) - 5) if vmin is None else vmin
    im = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, fmt.format(values[i, j]), ha="center", va="center", fontsize=7)
    ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=30, ha="right")
    ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8)
    return im


FAMILY_COLOR = {"CNN": "#4C72B0", "ViT": "#DD8452", "Embedding": "#55A868", "Generative VLM": "#C44E52",
                "Decision model": "#8172B3"}

res = pd.DataFrame(list(RESULTS.values()))
EIDS = [e for e in EXPS if e in PREDS]
print("Finished experiments:", EIDS)

## 6.1 Main results table
Metrics, 95% bootstrap CI of macro-F1, calibration (ECE before / after temperature scaling), speed.

In [ ]:
def build_results_table():
    t = res[res["eid"].isin(EIDS)].copy()
    ci = {e: bootstrap_ci(Y_TEST, PREDS[e]["y_pred"]) for e in t["eid"]}
    t["f1_ci_low"] = t["eid"].map(lambda e: ci[e][0])
    t["f1_ci_high"] = t["eid"].map(lambda e: ci[e][1])
    cols = ["eid", "name", "family", "init", "epochs", "params_M", "test_acc", "test_prec", "test_rec",
            "test_f1", "f1_ci_low", "f1_ci_high", "ece", "ece_ts", "temperature", "test_img_per_s",
            "peak_vram_gb", "train_s", "final_train_acc", "final_val_acc", "best_val_epoch",
            "invalid_rate", "order_flip_rate", "prompt"]
    return t[[c for c in cols if c in t.columns]].reset_index(drop=True)


TAB = show_table(build_results_table())
CI = {r["eid"]: (r["f1_ci_low"], r["f1_ci_high"]) for _, r in TAB.iterrows()}

## 6.2 Test macro-F1 with 95% bootstrap CIs

In [ ]:
def fig_test_f1():
    t = TAB.sort_values("test_f1")
    fig, ax = plt.subplots(figsize=(9, 0.42 * len(t) + 1.4))
    for i, r in enumerate(t.itertuples()):
        lo, hi = CI[r.eid]
        ax.barh(i, r.test_f1 * 100, color=FAMILY_COLOR[r.family],
                xerr=[[max(r.test_f1 - lo, 0) * 100], [max(hi - r.test_f1, 0) * 100]], capsize=3)
        ax.text(hi * 100 + 0.8, i, f"{r.test_f1 * 100:.1f}", va="center", fontsize=8)
    ax.set_yticks(range(len(t))); ax.set_yticklabels([r.name for r in t.itertuples()])
    ax.set_xlim(0, 108); ax.set_xlabel("test macro-F1 (%), error bar = 95% bootstrap CI")
    fams = [f for f in FAMILY_COLOR if f in set(t["family"])]
    ax.legend([plt.Rectangle((0, 0), 1, 1, color=FAMILY_COLOR[f]) for f in fams], fams,
              loc="lower right", fontsize=8)
    ax.set_title("Animals-10: test macro-F1 of the DL experiments")
    fig.tight_layout()
    return fig


show_figure(fig_test_f1())

## 6.3 Training curves of E1-E6
Top row: from scratch (20 epochs). Bottom row: fine-tuned from pretrained weights (5 epochs). No
augmentation, so the train-val gap shows over-fitting directly.

In [ ]:
def load_history(eid):
    return HISTORY.get(eid)


def fig_training_curves():
    groups = [("from scratch", [e for e in TRAINED if TRAINED[e]["init"] == "scratch"]),
              ("fine-tuned", [e for e in TRAINED if TRAINED[e]["init"] != "scratch"])]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
    for row, (gname, eids) in zip(axes, groups):
        for eid in eids:
            h = load_history(eid)
            if h is None:
                continue
            cfg = TRAINED[eid]
            kw = dict(color=f"C{list(TRAINED).index(eid)}", marker="o", ms=3,
                      label=f"{cfg['name']} ({len(h['epoch'])} ep)")
            ls = "-" if cfg["family"] == "CNN" else "--"
            ep = h["epoch"]
            row[0].plot(ep, h["train_loss"], ls, **kw)
            row[1].plot(ep, np.array(h["val_f1"]) * 100, ls, **kw)
            row[2].plot(ep, (np.array(h["train_acc"]) - np.array(h["val_acc"])) * 100, ls, **kw)
        for ax, t in zip(row, ["train loss", "validation macro-F1 (%)", "train - val accuracy (pp)"]):
            ax.set_title(f"{gname}: {t}"); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        row[2].axhline(0, color="black", lw=0.6)
        if row[0].get_legend_handles_labels()[0]:
            row[0].legend(fontsize=7)
    fig.tight_layout()
    return fig


if any(load_history(e) for e in TRAINED):
    show_figure(fig_training_curves())

## 6.4 Confusion matrices (row-normalized; E8's invalid answers are left out of the columns) and per-class F1

In [ ]:
def fig_confusions():
    n = len(EIDS)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.4 * rows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.axis("off")
    for ax, eid in zip(axes.flat, EIDS):
        confusion_panel(ax, PREDS[eid]["y_true"], PREDS[eid]["y_pred"], CLASS_NAMES,
                        f"{EXPS[eid]['name']}  (F1 {run_row(eid)['test_f1'] * 100:.1f}%)")
    fig.tight_layout()
    return fig


def fig_per_class():
    vals = np.stack([PREDS[e]["f1_per_class"] * 100 for e in EIDS])
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(EIDS) + 1.6))
    im = heatmap_panel(ax, vals, [EXPS[e]["name"] for e in EIDS], CLASS_NAMES)
    fig.colorbar(im, ax=ax, fraction=0.03, label="F1 (%)")
    ax.set_title("Per-class test F1 (%)")
    fig.tight_layout()
    return fig


show_figure(fig_confusions())
show_figure(fig_per_class())

## 6.5 Inference cost: latency, parameters, model size (20 test images, batch size 1)
For the trained models and E7 the image is preprocessed outside the timed region (model only); for E8
the whole generation and for E9 the two decision calls are timed.

In [ ]:
COST_N = 20                     # test images used to measure inference cost (same images as the classical notebook)
COST_WARMUP = 3                 # untimed calls before measuring
COST_REPEATS = 5                # timed calls per image for single-forward models; the median is kept
COST_REPEATS_GEN = 2            # timed calls per image for E8 / E9


class TrainedPredictor:
    def __init__(self, eid):
        ck = torch.load(model_path(eid), map_location=DEVICE)
        self.model = AnimalClassifier(ck["timm"], NUM_CLASSES, pretrained=False).to(DEVICE).eval()
        self.model.load_state_dict(ck["state_dict"])
        self.tf = get_transforms(ck["timm"])

    def tensor(self, it):
        return self.tf(item_image(it)).unsqueeze(0).to(DEVICE)


def load_predictor(eid):
    if eid in TRAINED:
        return TrainedPredictor(eid)
    return {"E7": ClipPredictor, "E8": QwenPredictor, "E9": ValenPredictor}[eid]()


def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def time_call(fn, repeats):
    """Median wall-clock seconds of fn() over `repeats` runs (GPU synchronised around each run)."""
    ts = []
    for _ in range(repeats):
        _sync()
        t0 = time.perf_counter()
        fn()
        _sync()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def pick_cost_images(test_idx, n, seed):
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(np.asarray(test_idx), n, replace=False)).astype(np.int64)


COST_IDX = pick_cost_images(TEST_IDX, COST_N, SEED + 20)
COST_WHAT = {"E7": "image encoder + cosine to 10 class texts", "E8": "full generation (vision + prompt + <=8 tokens)",
             "E9": "2 decision calls (2 option orders)"}


def module_size(m):
    """(params in millions, in-memory size of params + buffers in MB, dtype of the weights)."""
    ps, bs = list(m.parameters()), list(m.buffers())
    params_m = sum(p.numel() for p in ps) / 1e6
    size_mb = sum(t.numel() * t.element_size() for t in ps + bs) / 2 ** 20
    dtypes = sorted({str(p.dtype).replace("torch.", "") for p in ps})
    return params_m, size_mb, "/".join(dtypes)


@torch.no_grad()
def dl_latency(eid):
    pr = load_predictor(eid)
    if eid in TRAINED:
        xs = [pr.tf(Image.open(SAMPLES[i][0]).convert("RGB")).unsqueeze(0).to(DEVICE) for i in COST_IDX]
        calls = [lambda x=x: pr.model(x) for x in xs]
    elif eid == "E7":
        xs = [pr.preprocess(Image.open(SAMPLES[i][0]).convert("RGB")).unsqueeze(0).to(DEVICE) for i in COST_IDX]
        calls = [lambda x=x: pr.scale * F.normalize(pr.model.encode_image(x).float(), dim=-1) @ pr.W.T
                 for x in xs]
    elif eid == "E8":
        prompt = run_row("E8")["prompt"]
        calls = [lambda p=p: pr.ask(p, prompt) for p in zs_image_paths(COST_IDX)]
    else:
        calls = [lambda p=p: (pr.ask(p, VALEN_ORDERS["original"]), pr.ask(p, VALEN_ORDERS["shuffled"]))
                 for p in zs_image_paths(COST_IDX)]
    reps = COST_REPEATS_GEN if eid in ("E8", "E9") else COST_REPEATS
    for _ in range(COST_WARMUP):
        calls[0]()
    t = np.array([time_call(c, reps) for c in calls], dtype=np.float64)
    params_m, size_mb, dtype = module_size(pr.model)
    del pr; free_gpu()
    return t, params_m, size_mb, dtype


def build_dl_cost():
    device = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
    rows = []
    for eid in EIDS:
        t, params_m, size_mb, dtype = dl_latency(eid)
        ms = t * 1e3
        ck = model_path(eid)
        rows.append({"model": EXPS[eid]["name"], "eid": eid, "family": EXPS[eid]["family"],
                     "one_prediction": COST_WHAT.get(eid, "forward pass, 224x224"), "n_images": len(ms),
                     "device": device, "batch_size": 1,
                     "latency_ms_mean": ms.mean(), "latency_ms_std": ms.std(),
                     "latency_ms_median": np.median(ms), "latency_ms_p95": np.percentile(ms, 95),
                     "latency_ms_min": ms.min(), "latency_ms_max": ms.max(),
                     "images_per_s": 1e3 / ms.mean(),
                     "params_M": params_m, "weights_MB": size_mb, "weights_dtype": dtype,
                     "checkpoint_MB": os.path.getsize(ck) / 2 ** 20 if eid in TRAINED and os.path.exists(ck) else np.nan})
    return pd.DataFrame(rows)


_ = show_table(build_dl_cost())

# 7. Inference and grounding on the shared test images
Every model on the same 7 images: 6 random test images (one per class, seed `VIZ_SEED`; the same as the
classical notebook) and test.png, a deliberately confusing cat-butterfly hybrid with no label.

| Models | scores | grounding |
|---|---|---|
| E1-E3 ConvNeXt-T | class probability | **Grad-CAM** on the last stage (7x7) |
| E4, E5 ViT-S/16 | class probability | **transformer attribution** (Chefer et al., 2021, 14x14) |
| E6 ViT-S/16 DINOv3 | class probability | **raw CLS attention** of the last layer (14x14), not class-specific |
| E7 MobileCLIP2-S0 | cosine between the image and each class text | **similarity Grad-CAM** for the predicted class (8x8) |
| E8 Qwen3.5-0.8B | first answer token: top 10 over the whole vocabulary (log) | **occlusion** of the answer token |
| E9 Valen-Decision-1.5 | choice probability, original and shuffled option order | **occlusion** of the decision |

All models see the same 224x224 image (squashed like the CNN/ViT loader), so every map lines up with
the picture. Titles: ✓ green = correct, ✗ red = wrong, black = no label (test.png).

Occlusion (E8, E9): a 56x56 window filled with the image's mean colour slides over the image in steps of
28 px (7x7 positions, 49 extra forward passes per image). The score is the log-probability margin of the
predicted class over the next-best class; the map is how much hiding each region lowers it.

## 7.1 The shared test images and helpers

In [ ]:
def pick_test_images(test_idx, y_test, n_classes, n, seed):
    """n random test images, one per class while n <= n_classes, never the same image twice."""
    rng = np.random.default_rng(seed)
    if n <= n_classes:
        classes = rng.choice(n_classes, n, replace=False)
    else:
        classes = np.concatenate([rng.permutation(n_classes) for _ in range(math.ceil(n / n_classes))])[:n]
    out = []
    for c in classes:
        pool = test_idx[y_test == c]
        pool = pool[~np.isin(pool, out)]
        out.append(int(rng.choice(pool)))
    return np.array(out, dtype=np.int64)


DEMO_IDX = pick_test_images(TEST_IDX, Y_TEST, NUM_CLASSES, VIZ_N_TEST, VIZ_SEED)
CONFUSE_ID = "19UAsXlHKbs_wEB-vw2Px0ZkzIQhq13Uj"          # public file id of test.png
CONFUSE_PATH = os.path.abspath("./test.png")


def is_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False


def get_confusing_image():
    if not (os.path.exists(CONFUSE_PATH) and is_image(CONFUSE_PATH)):
        try:
            import gdown
        except ImportError:
            pip_install("gdown"); import gdown
        gdown.download(id=CONFUSE_ID, output=CONFUSE_PATH)
    if os.path.exists(CONFUSE_PATH) and is_image(CONFUSE_PATH):
        return CONFUSE_PATH
    if os.path.exists(CONFUSE_PATH):   # e.g. a Google Drive error page saved under test.png
        os.remove(CONFUSE_PATH)
    return None


def dataset_item(i):
    return {"idx": int(i), "path": SAMPLES[i][0], "true": int(LABELS[i]), "label": f"#{i} {CLASS_NAMES[LABELS[i]]}"}


DEMO_ITEMS = [dataset_item(i) for i in DEMO_IDX]
if get_confusing_image():
    DEMO_ITEMS.append({"idx": -1, "path": CONFUSE_PATH, "true": -1, "label": "test.png: cat-butterfly hybrid"})
else:
    print("WARNING: test.png could not be downloaded; upload it to /content and rerun this cell.")
print("Images:", [it["label"] for it in DEMO_ITEMS])

CLASS_COLORS = plt.cm.tab10(np.arange(NUM_CLASSES))


def item_image(it, size=224):
    return Image.open(it["path"]).convert("RGB").resize((size, size))


def item_color(pred, it):
    """green = correct, red = wrong, black = no ground truth."""
    return "black" if it["true"] < 0 else ("green" if pred == it["true"] else "red")


def items_key(items):
    """Short hash of an image list."""
    return hashlib.sha1("|".join(it["label"] for it in items).encode()).hexdigest()[:8]


def to_224(cam):
    t = torch.as_tensor(np.asarray(cam, dtype=np.float32))[None, None]
    up = F.interpolate(t, size=(224, 224), mode="bilinear", align_corners=False)[0, 0].numpy()
    up = up - up.min()
    return up / (up.max() + 1e-8)


def grad_cam(forward, layer, score_fn):
    """Grad-CAM (Selvaraju et al.): channel weights = mean gradient, map = ReLU(sum_c w_c * A_c).
    `layer` must output (B, C, H, W)."""
    store = {}

    def hook(_, __, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(hook)
    try:
        with torch.enable_grad():
            score_fn(forward()).backward()
    finally:
        h.remove()
    a, g = store["a"][0].detach().float(), store["g"][0].float()
    return F.relu((g.mean((1, 2), keepdim=True) * a).sum(0)).cpu().numpy()


def vit_relevance(model, x, cls):
    """Transformer attribution (Chefer, Gur & Wolf, 2021): per block Abar = mean_heads ReLU(dA * A),
    R <- R + Abar R from the identity. CLS-pooled models: CLS row of R; average-pooled: mean of patch rows."""
    bb = model.backbone
    attn, grads, hooks = [], [], []
    for blk in bb.blocks:
        blk.attn.fused_attn = False                 # materialize the attention matrix

        def hook(_, __, out):
            attn.append(out.detach())
            out.register_hook(lambda g: grads.append(g.detach()))
        hooks.append(blk.attn.attn_drop.register_forward_hook(hook))
    try:
        with torch.enable_grad():
            out = model(x)
            model.zero_grad(set_to_none=True)
            out[0, cls].backward()
    finally:
        for h in hooks:
            h.remove()
        for blk in bb.blocks:
            blk.attn.fused_attn = True
    grads = grads[::-1]                             # backward visits the blocks in reverse order
    n = attn[0].shape[-1]
    R = torch.eye(n, device=x.device)
    for A, G in zip(attn, grads):
        R = R + (G * A).clamp(min=0).mean(1)[0] @ R
    R = R - torch.eye(n, device=x.device)          # keep only relevance propagated through attention
    pre = bb.num_prefix_tokens
    side = int(round(math.sqrt(n - pre)))
    rel = R[0, pre:] if bb.global_pool == "token" else R[pre:, pre:].mean(0)
    return rel.reshape(side, side).cpu().numpy()


@torch.no_grad()
def vit_cls_attention(model, x):
    """Last-layer attention of the CLS token to the patches, averaged over heads (DINO-style map)."""
    bb = model.backbone
    blk, store = bb.blocks[-1], []
    blk.attn.fused_attn = False
    h = blk.attn.attn_drop.register_forward_hook(lambda _, __, out: store.append(out))
    try:
        model(x)
    finally:
        h.remove(); blk.attn.fused_attn = True
    pre = bb.num_prefix_tokens
    a = store[0][0, :, 0, pre:].mean(0)
    side = int(round(math.sqrt(a.numel())))
    return a.reshape(side, side).cpu().numpy()


OCC_WIN, OCC_STRIDE = 56, 28     # 7x7 overlapping window positions on the 224x224 image


def occlusion_images(img):
    a = np.asarray(img).copy()
    fill = a.reshape(-1, 3).mean(0).astype(np.uint8)
    out = []
    for y0 in range(0, a.shape[0] - OCC_WIN + 1, OCC_STRIDE):
        for x0 in range(0, a.shape[1] - OCC_WIN + 1, OCC_STRIDE):
            b = a.copy()
            b[y0:y0 + OCC_WIN, x0:x0 + OCC_WIN] = fill
            out.append((Image.fromarray(b), (y0, x0)))
    return out


def class_margin(lp, c):
    """log-probability margin of class c over the best other class."""
    return float(lp[c] - np.max(np.delete(lp, c)))


def occlusion_map(score_fn, img, c, tag, desc=""):
    """score_fn(path) -> 10 class log-probabilities. Returns a 224x224 map of the margin drop."""
    occ_dir = os.path.join(ZS_CACHE, "occlusion"); os.makedirs(occ_dir, exist_ok=True)
    p0 = os.path.join(occ_dir, f"{tag}_base.png"); img.save(p0)
    base = class_margin(score_fn(p0), c)
    heat, cnt = np.zeros((224, 224)), np.zeros((224, 224))
    for j, (im, (y0, x0)) in enumerate(tqdm(occlusion_images(img), desc=desc, leave=False)):
        p = os.path.join(occ_dir, f"{tag}_w{j}.png")
        im.save(p)
        heat[y0:y0 + OCC_WIN, x0:x0 + OCC_WIN] += base - class_margin(score_fn(p), c)
        cnt[y0:y0 + OCC_WIN, x0:x0 + OCC_WIN] += 1
    return np.clip(heat / np.maximum(cnt, 1), 0, None)


def item_zs_path(it, tag):
    """224x224 PNG of the item for E8/E9 (same resize as the CNN/ViT loader)."""
    p = os.path.join(ZS_CACHE, f"item_{tag}.png")
    item_image(it).save(p)
    return p

## 7.2 Per-model inference: probabilities + grounding map
Each model returns one record per image: `pred`, `probs` (10-class distribution), `bars` (what the
scores column draws) and `map` (grounding).

In [ ]:
KEEP_LOADED = False      # keep the last zero-shot model on the GPU between calls


_PREDICTORS = {}


def get_predictor(eid):
    """Load a model for inference (with KEEP_LOADED, the last loaded one is reused)."""
    if eid in _PREDICTORS:
        return _PREDICTORS[eid]
    _PREDICTORS.clear(); free_gpu()
    pr = load_predictor(eid)
    if KEEP_LOADED:
        _PREDICTORS[eid] = pr
    return pr


def done_with_predictor():
    if not KEEP_LOADED:
        free_gpu()


def map_name(eid):
    if eid in TRAINED:
        if TRAINED[eid]["family"] == "CNN":
            return "Grad-CAM (last stage, 7x7)"
        return "raw CLS attention, last layer (14x14)" if eid == "E6" else "transformer attribution (14x14)"
    return {"E7": "similarity Grad-CAM (8x8)", "E8": "occlusion (answer token)", "E9": "occlusion (decision)"}[eid]


def rows_trained(eid, items, tag):
    pr = get_predictor(eid)
    family = TRAINED[eid]["family"]
    rows = []
    for it in items:
        x = pr.tensor(it)
        with torch.no_grad():
            probs = torch.softmax(pr.model(x).float(), -1)[0].cpu().numpy()
        c = int(probs.argmax())
        if family == "CNN":
            cam = grad_cam(lambda: pr.model(x), pr.model.backbone.stages[-1], lambda o: o[0, c])
        elif eid == "E6":
            cam = vit_cls_attention(pr.model, x)
        else:
            cam = vit_relevance(pr.model, x, c)
        rows.append({"pred": c, "probs": probs, "conf": f"p {probs[c]:.2f}", "bars": ("prob", probs), "map": cam})
    del pr; done_with_predictor()
    return rows


def rows_e7(items, tag):
    clip = get_predictor("E7")
    W = clip.W.detach()

    def embed(x):
        return F.normalize(clip.model.encode_image(x).float(), dim=-1)

    rows = []
    for it in items:
        x = clip.preprocess(item_image(it)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            cos = (embed(x) @ W.T)[0].cpu().numpy()
        probs = softmax_np(clip.scale * cos[None])[0]
        c = int(cos.argmax())
        cam = grad_cam(lambda: embed(x), clip.model.visual.trunk.final_conv,
                       lambda e, c=c: (e[0] @ W[c]) - (e[0] @ W.T).mean())
        rows.append({"pred": c, "probs": probs, "conf": f"softmax p {probs[c]:.2f}", "bars": ("cos", cos),
                     "map": cam})
    del clip; done_with_predictor()
    return rows


def rows_e8(items, tag, topk=15):
    qwen = get_predictor("E8")
    prompt = run_row("E8")["prompt"]
    rows = []
    for r, it in enumerate(items):
        path = item_zs_path(it, f"{tag}_{r}")
        rec = qwen.ask_detailed(path, prompt, topk=topk)
        c = rec["pred"] if rec["pred"] >= 0 else int(np.argmax(rec["class_probs"]))
        amap = occlusion_map(lambda p: qwen.class_logprobs(p, prompt), item_image(it), c,
                             f"e8_{tag}_{r}", f"E8 occlusion {r + 1}/{len(items)}")
        rows.append({"pred": c, "probs": rec["class_probs"], "conf": f'answer "{rec["text"][:14]}"',
                     "bars": ("vocab", rec["first_topk"]), "map": amap, "answer": rec["text"]})
    del qwen; done_with_predictor()
    return rows


def rows_e9(items, tag):
    valen = get_predictor("E9")
    rows = []
    for r, it in enumerate(items):
        path = item_zs_path(it, f"{tag}_{r}")
        po, ps = valen.ask(path, VALEN_ORDERS["original"]), valen.ask(path, VALEN_ORDERS["shuffled"])
        c = int((po + ps).argmax())
        amap = occlusion_map(lambda p: log_probs(valen.ask(p)), item_image(it), c, f"e9_{tag}_{r}",
                             f"E9 occlusion {r + 1}/{len(items)}")
        rows.append({"pred": c, "probs": (po + ps) / 2, "conf": f"p {po[c]:.2f} / {ps[c]:.2f}",
                     "bars": ("two", po, ps), "map": amap})
    del valen; done_with_predictor()
    return rows


def compute_rows(eid, items, tag):
    """Inference records of model `eid` on `items`."""
    if eid in TRAINED:
        return rows_trained(eid, items, tag)
    return {"E7": rows_e7, "E8": rows_e8, "E9": rows_e9}[eid](items, tag)


def draw_mid(ax, it, row, eid, vocab_k=10, legend=True, fs=5.5):
    """Scores column: class probabilities (E1-E6), cosine to each class text (E7), first answer
    token over the whole vocabulary (E8), probabilities for the 2 option orders (E9)."""
    kind = row["bars"][0]
    y = np.arange(NUM_CLASSES)[::-1]
    if kind in ("prob", "cos"):
        v = np.asarray(row["bars"][1], dtype=float)
        cols = ["#55A868" if j == it["true"] else ("#C44E52" if j == row["pred"] else "#B0B0B0")
                for j in range(NUM_CLASSES)]
        ax.barh(y, v, color=cols, height=0.8)
        ax.set_yticks(y); ax.set_yticklabels(CLASS_NAMES, fontsize=fs)
        if kind == "cos":
            ax.set_xlim(v.min() - 0.02, v.max() + 0.02); xl = "cosine"
        else:
            ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); xl = "probability"
    elif kind == "vocab":
        ft = row["bars"][1][:vocab_k]
        yy = np.arange(len(ft))[::-1]
        ax.barh(yy, [f["prob"] for f in ft], height=0.8,
                color=[CLASS_COLORS[f["cls"]] if f["cls"] >= 0 else "#B0B0B0" for f in ft])
        ax.set_yticks(yy); ax.set_yticklabels([f["token"] for f in ft], fontsize=fs)
        ax.set_xscale("log"); ax.set_xlim(1e-5, 1.5); ax.minorticks_off()
        ax.set_xticks([1e-4, 1e-2, 1]); xl = "token prob. (log)"
    else:                                      # "two": E9, two option orders
        po, ps = row["bars"][1], row["bars"][2]
        ax.barh(y + 0.2, po, 0.4, color="#4C72B0", label="original order")
        ax.barh(y - 0.2, ps, 0.4, color="#DD8452", label="shuffled order")
        ax.set_yticks(y); ax.set_yticklabels(CLASS_NAMES, fontsize=fs)
        ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1]); xl = "choice probability"
        if legend:
            ax.legend(fontsize=fs - 0.5, loc="lower right", handlelength=0.8, borderaxespad=0.1)
    ax.tick_params(axis="both", labelsize=fs, length=2, pad=1)
    ax.set_xlabel(xl, fontsize=fs, labelpad=1)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    p = row["pred"]
    mark = "" if it["true"] < 0 else ("✓ " if p == it["true"] else "✗ ")
    name = CLASS_NAMES[p] if p >= 0 else "invalid"
    ax.set_title(f"{mark}{name} ({row['conf']})", fontsize=fs + 0.5, color=item_color(p, it), pad=2,
                 fontweight="normal")


def draw_map(ax, it, cam):
    ax.imshow(item_image(it))
    if cam is not None and float(np.max(cam)) > 0:
        ax.imshow(to_224(cam), cmap="jet", alpha=0.45)
    elif cam is not None:
        ax.text(112, 112, "no positive\nevidence", ha="center", va="center", fontsize=5, color="white",
                bbox=dict(facecolor="black", alpha=0.6, lw=0))
    ax.axis("off")


def draw_image(ax, it, fs=5.5):
    ax.imshow(item_image(it)); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    lab = it["label"] if it["true"] >= 0 else it.get("short", "test.png") + " (no label)"
    ax.set_title(lab, fontsize=fs + 0.5, pad=2, fontweight="normal")


def fig_rows(items, groups, title, row_h=1.12, scale=1.0, img_w=0.95, header_h=0.3, fs=5.5):
    """Rows = images; first column = the image, then `groups` of columns.
    groups = [(header, [(width, draw(ax, k)), ...])]; the header is centred over its columns."""
    widths = [img_w] + [w for _, cols in groups for w, _ in cols]
    n = len(items)
    H = header_h + row_h * n + 0.3
    fig = plt.figure(figsize=(sum(widths) * scale, H), layout="constrained")
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01, wspace=0.02, hspace=0.03)
    gs = fig.add_gridspec(n + 1, len(widths), width_ratios=widths, height_ratios=[header_h] + [row_h] * n)

    def header(cell, text):
        hax = fig.add_subplot(cell); hax.axis("off")
        hax.text(0.5, 0.0, text, ha="center", va="bottom", fontsize=fs + 1, fontweight="bold",
                 transform=hax.transAxes)

    header(gs[0, 0], "image")
    for k, it in enumerate(items):
        draw_image(fig.add_subplot(gs[k + 1, 0]), it, fs=fs)
    j = 1
    for head, cols in groups:
        header(gs[0, j:j + len(cols)], head)
        for c, (_, draw) in enumerate(cols):
            for k in range(n):
                draw(fig.add_subplot(gs[k + 1, j + c]), k)
        j += len(cols)
    fig.suptitle(title, fontsize=fs + 3.5, fontweight="bold")
    return fig


SHORT_NAME = {"E1": "ConvNeXt-T scratch", "E2": "ConvNeXt-T IN-22k", "E3": "ConvNeXt-T DINOv3",
              "E4": "ViT-S scratch", "E5": "ViT-S IN-21k", "E6": "ViT-S DINOv3", "E7": "MobileCLIP2-S0",
              "E8": "Qwen3.5-0.8B", "E9": "Valen-Dec.-0.8B"}
MID_HEADER = {"E7": "cosine: image vs each class text",
              "E8": "first answer token, top 10 of the vocabulary",
              "E9": "choice prob., 2 option orders"}


def method_short(eid):
    if eid in TRAINED:
        return "probability | " + ("Grad-CAM" if TRAINED[eid]["family"] == "CNN" else
                                   ("CLS attention" if eid == "E6" else "transformer attribution"))
    return {"E7": "cosine | similarity Grad-CAM", "E8": "first answer token | occlusion",
            "E9": "choice prob. (2 orders) | occlusion"}[eid]


def fig_models(items, dl_rows, eids, title, scale=1.1, row_h=1.12, fs=5.5):
    """Rows = images: image | scores | grounding, for the models `eids` side by side."""
    groups = []
    for eid in eids:
        if eid not in dl_rows:
            continue
        rows = dl_rows[eid]

        def mid(ax, k, eid=eid, rows=rows):
            draw_mid(ax, items[k], rows[k], eid, legend=(k == 0), fs=fs)

        def gmap(ax, k, rows=rows):
            draw_map(ax, items[k], rows[k]["map"])
        head = (f"{eid} {SHORT_NAME[eid]}\n{method_short(eid)}" if len(eids) > 1 else
                f"{MID_HEADER.get(eid, 'class probabilities')}\n| {map_name(eid)}")
        groups.append((head, [(1.35, mid), (0.95, gmap)]))
    return fig_rows(items, groups, title, scale=scale, row_h=row_h, fs=fs)


# One figure for the 6 trained models together, then one for each zero-shot model.
FIG_GROUPS = {"E1-E6": ("E1", "E2", "E3", "E4", "E5", "E6"), "E7": ("E7",), "E8": ("E8",), "E9": ("E9",)}


def group_title(gname, eids):
    if len(eids) == 1:
        return f"{eids[0]} {EXPS[eids[0]]['name']}"
    return f"{gname}: ConvNeXt-T and ViT-S/16 (scratch / supervised / DINOv3)"

## 7.3 Figures: E1-E6 together, then E7, E8, E9 (one row per image)

In [ ]:
DEMO_ROWS = {eid: compute_rows(eid, DEMO_ITEMS, "demo") for eid in EIDS}
for gname, eids in FIG_GROUPS.items():
    if any(e in DEMO_ROWS for e in eids):
        show_figure(fig_models(DEMO_ITEMS, DEMO_ROWS, eids, group_title(gname, eids)))